# CAM Medication Wastage 
**Model Building - LR Features**

STEP 1 - Load the packages

In [1]:
import pickle  # For saving and loading models
import os      # For file path operations
import time
import matplotlib.pyplot as plt #plotting

# Data Processing
import pandas as pd  # For data manipulation
import numpy as np   # For numerical operations
from sklearn.preprocessing import MinMaxScaler  # For scaling data

# Models
from sklearn import svm, linear_model, neighbors, tree, naive_bayes, ensemble
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV, RidgeClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic, DotProduct, Matern, WhiteKernel
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.ensemble import StackingClassifier

# Model Selection and Cross-Validation
from sklearn.model_selection import (
    RandomizedSearchCV,
    train_test_split, 
    cross_val_score, 
    StratifiedKFold, 
    cross_val_predict, 
    LeaveOneOut, 
    KFold
)
from sklearn.pipeline import make_pipeline  # For creating pipelines
from sklearn.utils.extmath import softmax

# Metrics
from sklearn.metrics import (
    roc_auc_score,
    roc_curve, 
    auc, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score, 
    balanced_accuracy_score, 
    precision_recall_curve, 
)
import statsmodels.api as sm
from statsmodels.stats.contingency_tables import mcnemar


STEP 2 - Load the data

In [2]:
# Load features metadata
allMeta_list = pd.read_csv('features_CAM_medAdherence.csv')
allMeta_sel_list = allMeta_list[allMeta_list['lr_fet'].isin(['other', 'yes'])]

feats_sel_list = allMeta_sel_list.loc[allMeta_sel_list['Types'] == "features", 'Varnames'].values.tolist()
print(feats_sel_list)
print("Features Count: %s" % len(feats_sel_list))

# Set categorical binary features
cat_bin_fet = allMeta_sel_list.loc[allMeta_sel_list['Cat_Cont_Ord'] == 'categorical_binary', 'Varnames'].tolist()
print("\nCategorical Binary Features: %s" % (cat_bin_fet))
print("Features Count: %s" % len(cat_bin_fet))

# Set categorical non-binary features
cat_nb_fet = allMeta_sel_list.loc[allMeta_sel_list['Cat_Cont_Ord'] == 'categorical_nonBinary', 'Varnames'].tolist()
print("\nCategorical Non Binary Features: %s" % (cat_nb_fet))
print("Features Count: %s" % len(cat_nb_fet))

# Set categorical features
cat_fet = allMeta_sel_list.loc[allMeta_sel_list['Cat_Cont_Ord'].isin(['categorical_binary','categorical_nonBinary']), 
                           'Varnames'].tolist()
print("\nCategorical Features: %s" % (cat_fet))
print("Features Count: %s" % len(cat_fet))

# Set continuous features
cont_fet = allMeta_sel_list.loc[allMeta_sel_list['Cat_Cont_Ord'] == 'continuous', 'Varnames'].tolist()
print("\nContinuous Features: %s" % (cont_fet))
print("Features Count: %s" % len(cont_fet))

['Race', 'Religion', 'Education', 'Hypertension', 'Medication', 'Dose', 'Duration', 'T_Nat', 'T_Hol']
Features Count: 9

Categorical Binary Features: ['Hypertension']
Features Count: 1

Categorical Non Binary Features: ['Race', 'Religion', 'Education']
Features Count: 3

Categorical Features: ['Race', 'Religion', 'Education', 'Hypertension']
Features Count: 4

Continuous Features: ['Medication', 'Dose', 'Duration', 'T_Nat', 'T_Hol']
Features Count: 5


In [3]:
# Load the data
train_data_ALL = pd.read_csv('CAM_MedAdherence_trainData_noNA_UNBALANCED.csv')
test_data_ALL = pd.read_csv('CAM_MedAdherence_testData_noNA.csv')

# Extract relevant features and outcome 
train_data = train_data_ALL[feats_sel_list + ['Adherence']]
test_data = test_data_ALL[feats_sel_list + ['Adherence']]


STEP 3 - Data Preprocessing

In [4]:
# Normalize continuous features : Min-max Normalization

# Fit the StandardScaler to the training data
zscore_scale = MinMaxScaler().fit(train_data[cont_fet])

# Transform the training and testing data
train_data_scaled = zscore_scale.transform(train_data[cont_fet])
test_data_scaled = zscore_scale.transform(test_data[cont_fet])

# Convert scaled arrays to DataFrames with float types and correct column names
train_scaled_df = pd.DataFrame(train_data_scaled, columns=cont_fet).astype(float)
test_scaled_df = pd.DataFrame(test_data_scaled, columns=cont_fet).astype(float)

# Drop the original columns to avoid dtype conflicts
train_data = train_data.drop(columns=cont_fet)
test_data = test_data.drop(columns=cont_fet)

# Concatenate the transformed data back to the original DataFrame
train_data = pd.concat([train_data, train_scaled_df], axis=1)
test_data = pd.concat([test_data, test_scaled_df], axis=1)


In [5]:
# Change categorical features to factor 
for col in cat_fet:
    train_data[col] = train_data[col].astype("category")
    test_data[col] = test_data[col].astype("category")
    
    # Check the category
    column_counts = test_data[col].value_counts()
    print(f"Column: {col}")
    print(column_counts)
    print()


Column: Race
Race
1    42
2    26
3    12
4     1
Name: count, dtype: int64

Column: Religion
Religion
1    43
2    22
4    10
3     5
5     1
Name: count, dtype: int64

Column: Education
Education
2    44
3    22
1    15
Name: count, dtype: int64

Column: Hypertension
Hypertension
2    51
1    30
Name: count, dtype: int64



In [6]:
# Outcome, 1 is Adherence while 0 is Non Adherence
train_data['Adherence'] = train_data['Adherence'].replace({1: 1, 2: 0})
test_data['Adherence'] = test_data['Adherence'].replace({1: 1, 2: 0})
print(test_data['Adherence'].value_counts())

Adherence
1    46
0    35
Name: count, dtype: int64


In [7]:
# Check if preprocessing done correctly
print(train_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189 entries, 0 to 188
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   Race          189 non-null    category
 1   Religion      189 non-null    category
 2   Education     189 non-null    category
 3   Hypertension  189 non-null    category
 4   Adherence     189 non-null    int64   
 5   Medication    189 non-null    float64 
 6   Dose          189 non-null    float64 
 7   Duration      189 non-null    float64 
 8   T_Nat         189 non-null    float64 
 9   T_Hol         189 non-null    float64 
dtypes: category(4), float64(5), int64(1)
memory usage: 10.4 KB
None


In [8]:
# Separate the features and outcome

# Features
X_train = train_data.drop('Adherence', axis=1)
X_test = test_data.drop('Adherence', axis=1)

# Outcome
Y_train = train_data['Adherence'].values.ravel()
Y_test = test_data['Adherence'].values.ravel()

print("X Train: %s" % ', '.join(map(str, X_train.shape)))
print("X_test: %s" % ', '.join(map(str, X_test.shape)))

X Train: 189, 9
X_test: 81, 9


STEP 4 - Model Building

In [9]:
# Base Models
modelIndi = [
    ('Logistic Regression', LogisticRegressionCV(max_iter=10000)),
    ('Ada Boost', AdaBoostClassifier(algorithm='SAMME')),
    ('Bagging', BaggingClassifier()),
    ('Gradient Boosting', GradientBoostingClassifier()),
    ('Random Forest', RandomForestClassifier(n_jobs=-1)),
    ('Gaussian Process', GaussianProcessClassifier()),
    ('SVM (Linear)', SVC(kernel='linear', probability=True)),
    ('SVM (Radial)', SVC(kernel='rbf', probability=True)),
    ('Decision Tree', DecisionTreeClassifier()),
    ('Bernoulli NB', BernoulliNB()),
    ('Gaussian NB', GaussianNB()),
    ('KNN', KNeighborsClassifier())
]

# Meta-learners 
modelMeta = [
   ('Ensemble GLM', LogisticRegression()),
   ('Ensemble RF',  RandomForestClassifier(n_jobs=-1)),
   ('Ensemble GBM', GradientBoostingClassifier()),
]


In [10]:
# Define the hyperparameter grid
param_grids_indi = {
    'Logistic Regression': {
        'Cs': [1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear'],
    },
    
    'Ada Boost': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0],
    },
    
    'Bagging': {
        'n_estimators': [10, 50, 100],
        'max_samples': [0.5, 0.7, 1.0],
        'max_features': [0.5, 0.7, 1.0],
    },
    
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.7, 1.0],
    },
    
    'Random Forest': {
        'n_estimators': [100, 200, 500],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False],
    },
    
    'Gaussian Process': {
        'kernel': [1*RBF(), 1*DotProduct(), 1*Matern(),  1*RationalQuadratic(), 1*WhiteKernel()],
    },
    
    'SVM (Linear)': {
        'C': [0.1, 1, 10],
    },
    
    'SVM (Radial)': {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
    },
    
    'Decision Tree': {
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 10, 20],
        'min_samples_leaf': [1, 2, 4],
    },
    
    'Bernoulli NB': {
        'alpha': [0.1, 1.0, 10.0],
        'binarize': [0.0, 0.5],
    },
    
    'Gaussian NB': {
        'var_smoothing': [1e-9, 1e-8, 1e-7],
    },
    
    'KNN': {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan'],
    }
}

param_grids_meta = {
    'Ensemble GLM': {
        'final_estimator__C': [0.1, 1, 10, 100],  # Regularization strength for LogisticRegression
        'final_estimator__penalty': ['l1', 'l2'],  # Penalty for LogisticRegression
        'final_estimator__solver': ['liblinear', 'saga'],
    },
    
    'Ensemble RF': {
        'final_estimator__n_estimators': [100, 200, 500],
        'final_estimator__max_depth': [5, 10, None],
        'final_estimator__min_samples_split': [2, 5, 10],
        'final_estimator__min_samples_leaf': [1, 2, 4],
        'final_estimator__bootstrap': [True, False],
    },
    
    'Ensemble GBM': {
        'final_estimator__n_estimators': [100, 200],
        'final_estimator__learning_rate': [0.01, 0.1, 0.2],
        'final_estimator__max_depth': [3, 5, 7],
        'final_estimator__subsample': [0.7, 1.0],
    }
}


In [11]:
perf_result = pd.DataFrame()
modelAll = [] #save the model
probAll = pd.DataFrame() #save the prediction probability
perf_result_idx = 0

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, 
                     shuffle=True, 
                     random_state=1234)


In [12]:
# Base models training & evaluation
for name, model in modelIndi:
    start_time = time.time()

    # Model names
    model_name = name
    perf_result.loc[perf_result_idx,'Models'] = model_name
    print(f'Model Name: {model_name}')

    # Hyperparameter tuning
    baseline_model = model
    baseline_model.fit(X_train, Y_train)
    randomized_search = RandomizedSearchCV(estimator=baseline_model, 
                                           param_distributions=param_grids_indi[model_name], 
                                           n_iter=15, 
                                           cv=cv,
                                           scoring='roc_auc',
                                           n_jobs=-1) # Create the RandomizedSearchCV object
    randomized_search.fit(X_train, Y_train)
    best_params_rand = randomized_search.best_params_ # Get the best hyperparameters
    best_model_rand = randomized_search.best_estimator_ # Get the best model
    modelAll.append((modelAll, best_model_rand))
    
    # Predictions and Probabilities
    test_pred = best_model_rand.predict(X_test)
    test_proba = best_model_rand.predict_proba(X_test)[:, 1]
    probAll[model_name] = test_proba

    # Confusion Matrix and Metrics
    cm = confusion_matrix(Y_test, test_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    ppv = tp / (tp + fp)
    npv = tn / (tn + fn)
    accuracy = accuracy_score(Y_test, test_pred)
    balanced_acc = balanced_accuracy_score(Y_test, test_pred)
    
    # ROC AUC and PR AUC
    mean_auc_train = round(randomized_search.cv_results_['mean_test_score'].mean(),3)
    roc_auc = roc_auc_score(Y_test, test_proba)
    precision, recall, _ = precision_recall_curve(Y_test, test_proba)
    pr_auc = auc(recall, precision)

    # Confidence Intervals for Accuracy and ROC AUC
    def confidence_interval(metric, n, z=1.96):
        se = np.sqrt((metric * (1 - metric)) / n)
        lower_bound = metric - z * se
        upper_bound = metric + z * se
        return lower_bound, upper_bound
    
    acc_lower, acc_upper = confidence_interval(accuracy, len(Y_test))
    roc_auc_lower, roc_auc_upper = confidence_interval(roc_auc, len(Y_test))

    # McNemar's Test
    mcnemar_result = mcnemar(cm)

    # Results
    perf_result.loc[perf_result_idx,'Training CV AUC'] = mean_auc_train
    perf_result.loc[perf_result_idx,'Testing CV AUC'] = f"{roc_auc:.3f} ({roc_auc_lower:.3f} - {roc_auc_upper:.3f})"
    perf_result.loc[perf_result_idx,'Testing Accuracy'] = f"{accuracy:.3f} ({acc_lower:.3f} - {acc_upper:.3f})"
    perf_result.loc[perf_result_idx,'Testing Sensitivity'] = round(specificity,3)
    perf_result.loc[perf_result_idx,'Testing Specificity'] = round(sensitivity,3)
    perf_result.loc[perf_result_idx,'Testing PPV'] = round(ppv,3)
    perf_result.loc[perf_result_idx,'Testing NPV'] = round(npv,3)
    perf_result.loc[perf_result_idx,'McNemar P-Value'] = round(mcnemar_result.pvalue,3)
    perf_result.loc[perf_result_idx,'Balanced Accuracy'] = round(balanced_acc,3)
    perf_result.loc[perf_result_idx,'PR-AUC'] = round(pr_auc,3)
    
    perf_result_idx+=1
    
    end_time = time.time()
    print(f'Time taken for evaluation(s): {end_time - start_time:.2f}')
    print(f'\n')
    

Model Name: Logistic Regression


c:\Users\user\anaconda3\envs\MLenv\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 6 is smaller than n_iter=15. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Time taken for evaluation(s): 22.10


Model Name: Ada Boost


c:\Users\user\anaconda3\envs\MLenv\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 9 is smaller than n_iter=15. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Time taken for evaluation(s): 5.66


Model Name: Bagging
Time taken for evaluation(s): 5.73


Model Name: Gradient Boosting
Time taken for evaluation(s): 13.55


Model Name: Random Forest
Time taken for evaluation(s): 27.58


Model Name: Gaussian Process


...

Time taken for evaluation(s): 0.26


Model Name: Decision Tree
Time taken for evaluation(s): 0.26


Model Name: Bernoulli NB
Time taken for evaluation(s): 0.15


Model Name: Gaussian NB


c:\Users\user\anaconda3\envs\MLenv\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 6 is smaller than n_iter=15. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\user\anaconda3\envs\MLenv\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 3 is smaller than n_iter=15. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Time taken for evaluation(s): 0.10


Model Name: KNN


c:\Users\user\anaconda3\envs\MLenv\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 12 is smaller than n_iter=15. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Time taken for evaluation(s): 0.29




In [13]:
# Meta models training & evaluation
for name, model in modelMeta:
    start_time = time.time()
    
    # Model names
    model_name = name
    perf_result.loc[perf_result_idx,'Models'] = model_name
    print(f'Model Name: {model_name}')

    # Hyperparameter tuning
    baseline_model = StackingClassifier(estimators=modelIndi, 
                                        final_estimator=model, 
                                        cv=cv)
    baseline_model.fit(X_train, Y_train)
    randomized_search = RandomizedSearchCV(estimator=baseline_model, 
                                           param_distributions=param_grids_meta[model_name], 
                                           n_iter=15, 
                                           cv=cv,
                                           scoring='roc_auc',
                                           n_jobs=-1) # Create the RandomizedSearchCV object
    randomized_search.fit(X_train, Y_train)
    best_params_rand = randomized_search.best_params_ # Get the best hyperparameters
    best_model_rand = randomized_search.best_estimator_ # Get the best model
    modelAll.append((modelAll, best_model_rand))
    
    # Predictions and Probabilities
    test_pred = best_model_rand.predict(X_test)
    test_proba = best_model_rand.predict_proba(X_test)[:, 1]
    probAll[model_name] = test_proba

    # Confusion Matrix and Metrics
    cm = confusion_matrix(Y_test, test_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    ppv = tp / (tp + fp)
    npv = tn / (tn + fn)
    accuracy = accuracy_score(Y_test, test_pred)
    balanced_acc = balanced_accuracy_score(Y_test, test_pred)
    
    # ROC AUC and PR AUC
    mean_auc_train = round(randomized_search.cv_results_['mean_test_score'].mean(),3)
    roc_auc = roc_auc_score(Y_test, test_proba)
    precision, recall, _ = precision_recall_curve(Y_test, test_proba)
    pr_auc = auc(recall, precision)

    # Confidence Intervals for Accuracy and ROC AUC
    acc_lower, acc_upper = confidence_interval(accuracy, len(Y_test))
    roc_auc_lower, roc_auc_upper = confidence_interval(roc_auc, len(Y_test))

    # McNemar's Test
    mcnemar_result = mcnemar(cm)

    # Results
    perf_result.loc[perf_result_idx,'Training CV AUC'] = mean_auc_train
    perf_result.loc[perf_result_idx,'Testing CV AUC'] = f"{roc_auc:.3f} ({roc_auc_lower:.3f} - {roc_auc_upper:.3f})"
    perf_result.loc[perf_result_idx,'Testing Accuracy'] = f"{accuracy:.3f} ({acc_lower:.3f} - {acc_upper:.3f})"
    perf_result.loc[perf_result_idx,'Testing Sensitivity'] = round(specificity,3)
    perf_result.loc[perf_result_idx,'Testing Specificity'] = round(sensitivity,3)
    perf_result.loc[perf_result_idx,'Testing PPV'] = round(ppv,3)
    perf_result.loc[perf_result_idx,'Testing NPV'] = round(npv,3)
    perf_result.loc[perf_result_idx,'McNemar P-Value'] = round(mcnemar_result.pvalue,3)
    perf_result.loc[perf_result_idx,'Balanced Accuracy'] = round(balanced_acc,3)
    perf_result.loc[perf_result_idx,'PR-AUC'] = round(pr_auc,3)
    
    perf_result_idx+=1
    
    end_time = time.time()
    print(f'Time taken for evaluation(s): {end_time - start_time:.2f}')
    print(f'\n')

Model Name: Ensemble GLM
Time taken for evaluation(s): 226.94


Model Name: Ensemble RF
Time taken for evaluation(s): 277.50


Model Name: Ensemble GBM
Time taken for evaluation(s): 225.96




In [14]:
# Save the results
import pickle  # For saving and loading models

# Save all models into a binary file 
with open('6_model.pkl', 'wb') as file:
    pickle.dump(modelAll, file)
    
# Save the probability
probAll['Actual'] = Y_test
probAll.to_csv("6_out_Ensemble_LRFet_unbalanced_resultProb.csv", index=False)

# Save the performance result
perf_result.to_csv("6_out_Ensemble_LRFet_unbalanced_resultPerf.csv", index=False)